# 01 — Environment Setup & Exploratory Data Analysis

This notebook:
1. Mounts Google Drive and sets project paths
2. Installs dependencies
3. Extracts the CIFAKE zip uploaded to Google Drive
4. Verifies class splits
5. Displays sample images
6. Computes per-channel statistics
7. Smoke-tests the DataLoader pipeline

**Prerequisites:** Upload the CIFAKE zip file to `My Drive/ai-image-detection/` before running Cell 3.

In [ ]:
# Cell 1 — Mount Google Drive & define paths
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
REPO_DIR = "/content/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone the repo (update URL to your own fork)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Extract CIFAKE from zip uploaded to Google Drive
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, "train")):
    print("CIFAKE already extracted — skipping.")
else:
    # Find the zip file in the Drive folder (handles any filename)
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            f"No zip file found in {DRIVE_ROOT}. "
            "Upload the CIFAKE zip downloaded from Kaggle to that folder."
        )

    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print("Extracting (this takes ~1-2 minutes)...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)

    print("Extraction complete.")

# Show what was extracted so we can verify the folder structure
extracted = os.listdir(DATA_DIR)
print(f"Contents of DATA_DIR: {extracted}")

# If zip extracted into a nested subfolder, adjust DATA_DIR automatically
if "train" not in extracted and len(extracted) == 1:
    DATA_DIR = os.path.join(DATA_DIR, extracted[0])
    print(f"Adjusted DATA_DIR to: {DATA_DIR}")

print(f"Final DATA_DIR contents: {os.listdir(DATA_DIR)}")

In [ ]:
# Cell 4 — Verify dataset splits
import os

splits = {}
for split in ["train", "test"]:
    for cls in ["REAL", "FAKE"]:
        path = os.path.join(DATA_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        splits[f"{split}/{cls}"] = count

print("CIFAKE Dataset Summary")
print("=" * 30)
for key, count in splits.items():
    print(f"  {key:15s} : {count:,}")
print(f"  {'Total':15s} : {sum(splits.values()):,}")

assert splits["train/REAL"] == 50_000 or splits["train/REAL"] == 30_000, (
    f"Unexpected train/REAL count: {splits['train/REAL']}"
)

In [ ]:
# Cell 5 — Sample image grid (REAL vs FAKE)
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for row, cls in enumerate(["REAL", "FAKE"]):
    folder = os.path.join(DATA_DIR, "train", cls)
    files = random.sample(os.listdir(folder), 4)
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(folder, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis("off")

fig.suptitle("CIFAKE Samples: REAL (top) vs FAKE (bottom)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Per-channel mean and std
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

raw_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])
raw_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=raw_transform)
raw_loader = DataLoader(raw_dataset, batch_size=256, shuffle=False, num_workers=2)

channel_sum = np.zeros(3)
channel_sq_sum = np.zeros(3)
n_pixels = 0

for images, _ in raw_loader:
    batch = images.numpy()
    channel_sum += batch.sum(axis=(0, 2, 3))
    channel_sq_sum += (batch ** 2).sum(axis=(0, 2, 3))
    n_pixels += batch.shape[0] * batch.shape[2] * batch.shape[3]

mean = channel_sum / n_pixels
std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)

print(f"CIFAKE Training Set Statistics (224x224)")
print(f"  Mean (R, G, B): ({mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f})")
print(f"  Std  (R, G, B): ({std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f})")
print(f"  (Using ImageNet normalization for transfer learning)")

In [ ]:
# Cell 7 — DataLoader smoke test
import sys
import numpy as np
sys.path.insert(0, REPO_DIR)

from src.utils.data_loader import get_cifake_loaders, get_dataset_stats

train_loader, val_loader, test_loader = get_cifake_loaders(DATA_DIR, batch_size=32)

for name, loader in [("Train", train_loader), ("Val", val_loader), ("Test", test_loader)]:
    stats = get_dataset_stats(loader)
    print(f"{name:5s} — samples: {stats['total_samples']:,}  "
          f"image shape: {stats['image_shape']}  "
          f"classes: {stats['class_to_idx']}")

images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Label distribution in batch: {dict(zip(*np.unique(labels.numpy(), return_counts=True)))}")
print("\nDataLoader pipeline is working.")